# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG

**Thời lượng:** 120 phút  
**Môi trường:** Google Colab (T4 GPU khuyến nghị) + Neo4j AuraDB  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = 1500`
- `LAB_MAX_CHUNKS = 3000`
- `EXTRACTION_MAX_CHUNKS = 400`

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

# PHẦN 1 — SETUP & PREPROCESSING

### Secrets trên Colab
Tạo:
- `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`
- `GROQ_API_KEY`, `GROQ_MODEL`
- `HF_TOKEN` để stream dataset từ Hugging Face
- cho judge: `JUDGE_PROVIDER`, `JUDGE_MODEL`, và `OPENAI_API_KEY` nếu dùng OpenAI

Không hard-code API key vào notebook nộp bài.

In [1]:
#@title 1.1 — Install
%pip -q install neo4j pandas numpy pyarrow sentence-transformers faiss-cpu groq openai tqdm networkx spacy datasets langchain-community llama-index

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
from dotenv import load_dotenv
load_dotenv(override=True)

#@title 1.2 — Imports & config
import os, re, json, time, random, hashlib, unicodedata
from pathlib import Path
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value is not None:
            return value
    except Exception:
        pass
    if name == "NEO4J_USER" and "NEO4J_USERNAME" in os.environ:
        return os.environ.get("NEO4J_USERNAME")
    return os.environ.get(name, default)

NEO4J_URI = get_secret("NEO4J_URI", "")
NEO4J_USER = get_secret("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", "neo4j")

GROQ_API_KEY = get_secret("GROQ_API_KEY", "")
GROQ_MODEL = get_secret("GROQ_MODEL", "openai/gpt-oss-20b")
GROQ_MODEL_FALLBACKS = [
    m.strip() for m in get_secret("GROQ_MODEL_FALLBACKS", "openai/gpt-oss-120b,qwen/qwen3.6-27b,groq/compound-mini,groq/compound,allam-2-7b").split(",")
    if m.strip()
]

# Schema Knowledge Graph — single source of truth (dùng chung cho extraction M2 và seed matching M4,
# tránh NameError khi các cell chạy không theo đúng thứ tự trên/dưới)
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS = {
    "ACQUIRED", "DEVELOPED", "INVESTED_IN", "FOUNDED",
    "WORKED_AT", "PARTNERED_WITH", "USES", "LEADS"
}

JUDGE_PROVIDER = get_secret("JUDGE_PROVIDER", "openai").lower()
JUDGE_MODEL = get_secret("JUDGE_MODEL", "openai/gpt-oss-20b")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY", "")
HF_TOKEN = get_secret("HF_TOKEN", "")

DATA_PATH = "hackernoon_subset.csv"
LAB_MAX_ARTICLES = 5000
LAB_MAX_CHUNKS = 6000
EXTRACTION_MAX_CHUNKS = 400
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40


## 1.3 — Download HackerNoon Dataset bằng Hugging Face Streaming

Cell dưới đây stream trực tiếp dataset **`HackerNoon/tech-company-news-data-dump`** và ghi dần ra CSV, nên không cần tải toàn bộ dataset vào RAM.

### Hai cơ chế giới hạn

- `LIMIT_ROWS`: số dòng tối đa.
- `LIMIT_MB`: dung lượng file tối đa.
- `PRIORITIZE_MB = True`: ưu tiên dừng theo dung lượng MB.
- `PRIORITIZE_MB = False`: thanh tiến trình theo số dòng, nhưng **vẫn giữ hard-stop `LIMIT_ROWS`**.

### Lưu ý

- Đặt `HF_TOKEN` trong **Colab Secrets**, không hard-code token vào notebook.
- Nếu dataset yêu cầu quyền truy cập/gated access, hãy mở trang dataset trên Hugging Face và hoàn tất bước **Agree/Request access** trước.
- Sau khi cell hoàn tất, `DATA_PATH` mặc định đã trỏ tới `/content/hackernoon_subset.csv`, nên cell loader kế tiếp có thể chạy trực tiếp.

In [ ]:
#@title 1.3 — Stream HackerNoon dataset -> CSV
import csv
import os
from datasets import load_dataset
from tqdm.auto import tqdm

DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
OUTPUT_CSV = "hackernoon_subset.csv"

# Giới hạn cho bản lab. Có thể tăng sau buổi học.
LIMIT_ROWS = 1_000_000
LIMIT_MB = 300

# True  -> progress/dừng ưu tiên theo MB
# False -> progress theo rows; vẫn có hard-stop LIMIT_ROWS
PRIORITIZE_MB = True

# Đọc từ Colab Secrets qua get_secret() ở cell config.
if not HF_TOKEN:
    raise ValueError(
        "Thiếu HF_TOKEN. Hãy thêm Hugging Face Access Token vào Colab Secrets với tên HF_TOKEN."
    )

print("Đang kết nối luồng dữ liệu (streaming)...")

try:
    dataset = load_dataset(
        DATASET_NAME,
        split="train",
        streaming=True,
        token=HF_TOKEN,
    )
    iterator = iter(dataset)

    first_row = next(iterator)
    headers = list(first_row.keys())

    print(f"Đang ghi dữ liệu vào: {OUTPUT_CSV}")

    rows_written = 0
    total_progress = LIMIT_MB if PRIORITIZE_MB else LIMIT_ROWS
    unit_progress = "MB" if PRIORITIZE_MB else "row"

    with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
        writer.writeheader()
        writer.writerow(first_row)
        rows_written += 1

        # Flush để kích thước file phản ánh dữ liệu vừa ghi.
        f.flush()
        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)

        with tqdm(
            total=total_progress,
            desc=f"Đang tải ({unit_progress})",
            unit=unit_progress,
        ) as pbar:
            if PRIORITIZE_MB:
                pbar.n = min(file_size_mb, LIMIT_MB)
                pbar.refresh()
            else:
                pbar.update(1)

            for row in iterator:
                writer.writerow(row)
                rows_written += 1

                # Kiểm tra dung lượng định kỳ để giảm overhead I/O.
                # Khi gần LIMIT_MB, kiểm tra mỗi row để dừng sát ngưỡng hơn.
                should_check_size = (
                    PRIORITIZE_MB
                    and (
                        rows_written % 100 == 0
                        or file_size_mb >= LIMIT_MB * 0.95
                    )
                )

                if should_check_size:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    pbar.n = min(round(file_size_mb, 2), LIMIT_MB)
                    pbar.refresh()
                elif not PRIORITIZE_MB:
                    pbar.update(1)

                # Hard-stop theo MB nếu đang ưu tiên dung lượng.
                if PRIORITIZE_MB and file_size_mb >= LIMIT_MB:
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn dung lượng: "
                        f"{file_size_mb:.2f} MB "
                        f"(Tổng: {rows_written:,} dòng)"
                    )
                    break

                # Hard-stop theo số dòng trong mọi chế độ.
                if rows_written >= LIMIT_ROWS:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn số dòng: "
                        f"{rows_written:,} dòng "
                        f"(Dung lượng: {file_size_mb:.2f} MB)"
                    )
                    break

        f.flush()

    final_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
    print(
        f"✅ Hoàn thành: {os.path.abspath(OUTPUT_CSV)}\n"
        f"   Rows: {rows_written:,}\n"
        f"   Size: {final_size_mb:.2f} MB"
    )

    # Đồng bộ đường dẫn cho cell loader tiếp theo.
    DATA_PATH = OUTPUT_CSV

except StopIteration:
    raise RuntimeError("Dataset stream rỗng: không lấy được dòng đầu tiên.")
except Exception as e:
    print(f"\n❌ Có lỗi xảy ra: {e}")
    print(
        "Kiểm tra: (1) HF_TOKEN, (2) quyền Agree/Access trên Hugging Face, "
        "(3) kết nối mạng của Colab."
    )
    raise

Đang kết nối luồng dữ liệu (streaming)...
Đang ghi dữ liệu vào: hackernoon_subset.csv


Đang tải (MB):   0%|          | 0/300 [00:00<?, ?MB/s]

In [4]:
#@title 1.4 — Neo4j connection + schema
driver = None

def connect_neo4j():
    global driver
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Thiếu Neo4j secrets.")
    if driver is not None:
        try:
            driver.close()
        except Exception:
            pass
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
        max_connection_lifetime=60,
        keep_alive=True
    )
    driver.verify_connectivity()
    print("✅ Neo4j connected.")

def run_cypher(query, **params):
    global driver
    if driver is None:
        connect_neo4j()
    for attempt in range(2):
        try:
            with driver.session(database=NEO4J_DATABASE) as session:
                result = session.run(query, **params)
                rows = [r.data() for r in result]
                result.consume()
            return rows
        except Exception as e:
            err_str = str(e).lower()
            if attempt == 0 and any(k in err_str for k in ["routing", "serviceunavailable", "closed", "connection", "defunct", "unavailable"]):
                print("⚠️ [Neo4j Reconnect] Kết nối bị ngắt do idle timeout, đang tự động kết nối lại...")
                connect_neo4j()
                continue
            raise

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print("✅ Schema ready.")

connect_neo4j()
setup_graph_schema()


✅ Neo4j connected.
✅ Schema ready.


In [5]:
#@title 1.5 — Loader + exact dedup + chunking
def norm_space(x):
    return re.sub(r"\s+", " ", str(x or "")).strip()

def sha1(x):
    return hashlib.sha1(str(x).encode("utf-8", errors="ignore")).hexdigest()

def pick_col(df, candidates, required=True):
    lookup = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    if required:
        raise KeyError(f"Missing one of columns: {candidates}")
    return None

def load_news(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported: {path.suffix}")

def standardize_news(raw):
    text_col = pick_col(raw, ["text", "content", "article", "body", "story", "description"])
    title_col = pick_col(raw, ["title", "headline"], required=False)
    date_col = pick_col(raw, ["published_date", "date", "published_at", "created_at"], required=False)
    id_col = pick_col(raw, ["id", "article_id", "story_id", "uuid"], required=False)

    df = pd.DataFrame()
    df["text"] = raw[text_col].fillna("").map(norm_space)
    df["title"] = raw[title_col].fillna("").map(norm_space) if title_col else ""

    if date_col:
        df["published_date"] = (
            pd.to_datetime(raw[date_col], errors="coerce", utc=True)
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    else:
        df["published_date"] = ""

    if id_col:
        df["article_id"] = raw[id_col].astype(str)
    else:
        df["article_id"] = [
            sha1(f"{t}\n{x}")[:20] for t, x in zip(df["title"], df["text"])
        ]

    df = df[df["text"].str.len() >= 80].copy()
    df["dedup_key"] = [
        sha1(norm_space(f"{t}\n{x}").lower())
        for t, x in zip(df["title"], df["text"])
    ]
    before = len(df)
    df = df.drop_duplicates("dedup_key").drop(columns="dedup_key").reset_index(drop=True)
    print(f"Exact dedup: {before:,} -> {len(df):,}")

    if LAB_MAX_ARTICLES and len(df) > LAB_MAX_ARTICLES:
        df = df.head(LAB_MAX_ARTICLES).reset_index(drop=True)
    return df

def chunk_text(text, size=220, overlap=40):
    words = norm_space(text).split()
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        part = words[start:start+size]
        if not part:
            break
        out.append(" ".join(part))
        if start + size >= len(words):
            break
    return out

def build_chunks(news_df):
    rows = []
    for r in tqdm(news_df.itertuples(index=False), total=len(news_df), desc="Chunking"):
        for i, text in enumerate(chunk_text(r.text, CHUNK_WORDS, CHUNK_OVERLAP_WORDS)):
            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "title": r.title,
                "published_date": r.published_date,
                "text": text,
            })
            if LAB_MAX_CHUNKS and len(rows) >= LAB_MAX_CHUNKS:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

raw_df = load_news(DATA_PATH)
news_df = standardize_news(raw_df)
chunks_df = build_chunks(news_df)
display(chunks_df.head())

Exact dedup: 47,284 -> 42,790


Chunking:   0%|          | 0/5000 [00:00<?, ?it/s]

,chunk_id,article_id,title,published_date,text
0,1a05beb7aa3071be6fd7::c0000,1a05beb7aa3071be6fd7,onsemi and Sineng Electric Spearhead the Development of Sustainable Energy Applications,2023-05-16,(Nasdaq: ON) a leader in intelligent power and sensing technologies today announced that Sineng Electric will integr...
1,ec98609611765e97440d::c0000,ec98609611765e97440d,Adobe student receives national Information and Technology award,2023-05-02,ELKO — An eighth grader at Adobe Middle School is one of 34 middle school aged girls in Nevada to be recognized by t...
2,8e922bc62b578e73e815::c0000,8e922bc62b578e73e815,Modernizing State Services: Harnessing Technology for Enhanced Public Service Delivery,2023-05-01,To deliver 21st-century government services Governors and cabinet members need leaders with technology expertise to ...
3,4bd7afdba71243b0dbcd::c0000,4bd7afdba71243b0dbcd,Terry Richardson On Why He Left AMD GreenPages’ Technology Chops And The AI Opportunity,2023-05-02,In February GreenPages acquired Toronto-based Zanaris an IT automation cloud and DevOps services firm ... Steve Burk...
4,e1dfbe88dae03136f847::c0000,e1dfbe88dae03136f847,Synex Renewable Energy Corporation (Formerly Synex International Inc.) Third Quarter of Fiscal 2023,2023-05-15,The conference will bring together growth oriented publicly traded clean energy and technology companies ... up to 4...


### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [42]:
#@title 1.6 — LLM wrapper có retry + JSON parsing + Fail-close
from groq import Groq
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

# Model biet da het han muc trong phien nay -> bo qua thang, khoi goi API vo ich
_EXHAUSTED_MODELS = set()

def parse_json_object(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    # Mot so model reasoning (vd qwen) xi khoi <think>...</think> ra truoc JSON that su
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S | re.I).strip()
    a = text.find("{")
    if a < 0:
        raise ValueError(f"No JSON object found in response: {text[:200]}")
    try:
        obj, _ = json.JSONDecoder().raw_decode(text[a:])
        return obj
    except json.JSONDecodeError as e:
        raise ValueError(f"No JSON object found in response: {text[:200]}") from e

def groq_chat(messages, model=None, json_mode=False, max_retries=4):
    if groq_client is None:
        raise RuntimeError("\u274c [FAIL-CLOSE] Thieu GROQ_API_KEY. Hay cau hinh trong file .env hoac secrets.")
    primary_model = model or os.environ.get("GROQ_MODEL", GROQ_MODEL)
    if not primary_model:
        raise RuntimeError("\u274c [FAIL-CLOSE] Thieu GROQ_MODEL.")

    # Model chain: model chinh truoc, roi cac model fallback (bo trung, giu thu tu)
    full_chain = [primary_model] + [m for m in GROQ_MODEL_FALLBACKS if m != primary_model]
    model_chain = [m for m in full_chain if m not in _EXHAUSTED_MODELS] or full_chain

    last = None
    for cur_model in model_chain:
        for attempt in range(max_retries):
            try:
                kwargs = {
                    "model": cur_model,
                    "messages": messages,
                    "temperature": 0.0,
                }
                if json_mode:
                    kwargs["response_format"] = {"type": "json_object"}

                resp = groq_client.chat.completions.create(**kwargs)
                content = resp.choices[0].message.content
                if json_mode:
                    parse_json_object(content)  # neu khong parse duoc -> coi nhu model nay that bai, roi vao except de retry/doi model
                usage = {}
                if getattr(resp, "usage", None):
                    usage = {
                        "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                        "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                        "total_tokens": getattr(resp.usage, "total_tokens", None),
                    }
                return content, usage
            except Exception as e:
                last = e
                err_str = str(e).lower()

                # Auto-fallback neu gap loi kiem duyet cu phap JSON cua Groq (json_validate_failed)
                if json_mode and ("json_validate_failed" in err_str or "failed to validate json" in err_str):
                    try:
                        kwargs_fallback = {
                            "model": cur_model,
                            "messages": messages,
                            "temperature": 0.0,
                        }
                        resp = groq_client.chat.completions.create(**kwargs_fallback)
                        content = resp.choices[0].message.content
                        parse_json_object(content)
                        usage = {}
                        if getattr(resp, "usage", None):
                            usage = {
                                "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                                "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                                "total_tokens": getattr(resp.usage, "total_tokens", None),
                            }
                        return content, usage
                    except Exception as fb_err:
                        last = fb_err
                        err_str = str(fb_err).lower()

                # Sai API key: fail-close ngay, doi model khong giai quyet duoc
                if "invalid_api_key" in err_str or re.search(r"\b401\b", err_str):
                    raise RuntimeError(f"\u274c [FAIL-CLOSE] API Key khong hop le (401): {e}") from e

                # Model nay het han muc / khong kha dung -> chuyen sang model ke tiep trong chain
                model_exhausted = (
                    "tokens per day" in err_str or "tpd" in err_str
                    or ("rate_limit_exceeded" in err_str and "tpd" in err_str)
                    or "model_decommissioned" in err_str
                    or "does not exist" in err_str
                    or "model_not_found" in err_str
                )
                if model_exhausted:
                    _EXHAUSTED_MODELS.add(cur_model)
                    print(f"\u26a0\ufe0f [Model switch] '{cur_model}' het han muc/khong kha dung: {e}. Chuyen sang model ke tiep trong chain...")
                    break  # thoat vong retry cua model nay, sang model ke tiep

                if attempt == max_retries - 1:
                    break
                wait_time = min(20, 2**attempt + random.random())
                print(f"\u26a0\ufe0f [Retry {attempt+1}/{max_retries}] Loi tam thoi ({cur_model}): {e}. Dang cho {wait_time:.1f}s...")
                time.sleep(wait_time)

    raise RuntimeError(f"\u274c [FAIL-CLOSE] Goi LLM that bai sau khi thu toan bo model chain {model_chain}: {last}")

def groq_json(system, user, model=None):
    text, usage = groq_chat(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        model=model,
        json_mode=True,
    )
    return parse_json_object(text), usage


## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [46]:
#@title 1.7 — Coreference resolution theo batch
COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
Resolve pronouns and generic references only when the antecedent is clearly supported in the same chunk.
Never invent facts. Preserve dates, numbers, tickers and product names.
Return strict JSON only.
""".strip()

def resolve_coref_batch(batch_df):
    payload = [{"chunk_id": r.chunk_id, "text": r.text}
               for r in batch_df.itertuples(index=False)]

    prompt = f"""
Resolve coreferences.

Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "resolved_text": "...",
      "unresolved_mentions": ["..."]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()

    obj, usage = groq_json(COREF_SYSTEM, prompt)
    by_id = {x.get("chunk_id"): x for x in obj.get("items", [])}

    rows = []
    for r in batch_df.itertuples(index=False):
        item = by_id.get(r.chunk_id, {})
        rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": norm_space(item.get("resolved_text") or r.text),
            "unresolved_mentions": item.get("unresolved_mentions", []),
        })
    return pd.DataFrame(rows), usage

def run_coref(chunks_subset, batch_size=10, max_consecutive_errors=3,
              ckpt_path="outputs/coref_checkpoint.csv"):
    consecutive_errors = 0
    out = []
    done_ids = set()
    if os.path.exists(ckpt_path):
        prev = pd.read_csv(ckpt_path)
        if not prev.empty:
            out.append(prev)
            done_ids = set(prev["chunk_id"].tolist())
            print(f"🔄 Đã tìm thấy checkpoint coref: {len(done_ids)}/{len(chunks_subset)} chunk đã xong. Đang chạy tiếp...")

    remaining = chunks_subset[~chunks_subset["chunk_id"].isin(done_ids)].reset_index(drop=True)

    for start in tqdm(range(0, len(remaining), batch_size), desc="Coref"):
        batch = remaining.iloc[start:start+batch_size]
        try:
            df, _ = resolve_coref_batch(batch)
            consecutive_errors = 0
        except Exception as e:
            consecutive_errors += 1
            if consecutive_errors >= max_consecutive_errors:
                if out:
                    pd.concat(out, ignore_index=True).to_csv(ckpt_path, index=False)
                raise RuntimeError(f"❌ [FAIL-CLOSE] Dừng Coreference sau {consecutive_errors} batches lỗi liên tiếp: {e}") from e
            df = pd.DataFrame({
                "chunk_id": batch["chunk_id"].tolist(),
                "resolved_text": batch["text"].tolist(),
                "unresolved_mentions": [["COREF_BATCH_FAILED"] for _ in range(len(batch))],
            })
        out.append(df)
        pd.concat(out, ignore_index=True).to_csv(ckpt_path, index=False)
    return pd.concat(out, ignore_index=True)

extraction_source = chunks_df.head(EXTRACTION_MAX_CHUNKS).copy()
coref_df = run_coref(extraction_source)
extraction_source = extraction_source.merge(coref_df, on="chunk_id", how="left")


Coref:   0%|          | 0/40 [00:00<?, ?it/s]

# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph schema
**Nodes:** `Company`, `Person`, `Technology` + base label `Entity`.

**Relations:** `ACQUIRED`, `DEVELOPED`, `INVESTED_IN`, `FOUNDED`, `WORKED_AT`, `PARTNERED_WITH`, `USES`, `LEADS`.

**Mỗi edge bắt buộc:** `source_chunk_id`, `published_date`; khuyến nghị thêm `evidence`, `confidence`.

> Relation type phải qua allowlist trước khi ghép vào Cypher.

In [16]:
#@title 2.1 — NER + RE extraction
# ALLOWED_NODE_TYPES / ALLOWED_RELATIONS: dinh nghia o Cell 1.2 (Imports & Config)
EXTRACT_SYSTEM = f"""
Extract a high-precision knowledge graph from tech-news text.
Allowed node types: {sorted(ALLOWED_NODE_TYPES)}
Allowed relations: {sorted(ALLOWED_RELATIONS)}
Use only explicitly supported facts. Prefer precision over recall.
Every relation needs short evidence. Return strict JSON only.
""".strip()

def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "published_date": r.published_date,
        "text": getattr(r, "resolved_text", None) or r.text,
    } for r in batch_df.itertuples(index=False)]

    prompt = f"""
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "relations": [
        {{
          "source": "...",
          "source_type": "Company|Person|Technology",
          "relation": "ALLOWED_RELATION",
          "target": "...",
          "target_type": "Company|Person|Technology",
          "evidence": "...",
          "confidence": 0.0
        }}
      ]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    return groq_json(EXTRACT_SYSTEM, prompt)

def run_extraction(source_df, batch_size=6, max_consecutive_errors=3,
                    ckpt_triples_path="outputs/extraction_triples_checkpoint.csv",
                    ckpt_state_path="outputs/extraction_progress.json"):
    consecutive_errors = 0
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()

    triples, errors = [], []
    done_starts = set()
    if os.path.exists(ckpt_state_path):
        with open(ckpt_state_path, "r", encoding="utf-8") as f:
            done_starts = set(json.load(f).get("done_starts", []))
    if os.path.exists(ckpt_triples_path) and done_starts:
        triples = pd.read_csv(ckpt_triples_path).to_dict("records")

    all_starts = list(range(0, len(source_df), batch_size))
    remaining_starts = [s for s in all_starts if s not in done_starts]
    if done_starts:
        print(f"🔄 Đã tìm thấy checkpoint extraction với {len(done_starts)}/{len(all_starts)} batch đã xong. Đang chạy tiếp các batch còn lại...")

    for start in tqdm(remaining_starts, desc="NER+RE"):
        batch = source_df.iloc[start:start+batch_size]
        try:
            obj, _ = extract_batch(batch)
            consecutive_errors = 0
        except Exception as e:
            errors.append({"start": start, "error": str(e)})
            consecutive_errors += 1
            if consecutive_errors >= max_consecutive_errors:
                raise RuntimeError(f"❌ [FAIL-CLOSE] Dừng Extraction sau {consecutive_errors} batches lỗi liên tiếp. Chi tiết: {e}") from e
            continue

        for item in obj.get("items", []):
            cid = item.get("chunk_id")
            if cid not in meta:
                continue
            for x in item.get("relations", []):
                s, t = norm_space(x.get("source")), norm_space(x.get("target"))
                st, tt, rel = x.get("source_type"), x.get("target_type"), x.get("relation")
                if not s or not t:
                    continue
                if st not in ALLOWED_NODE_TYPES or tt not in ALLOWED_NODE_TYPES:
                    continue
                if rel not in ALLOWED_RELATIONS:
                    continue
                triples.append({
                    "source_raw": s,
                    "source_type": st,
                    "relation": rel,
                    "target_raw": t,
                    "target_type": tt,
                    "source_chunk_id": cid,
                    "published_date": meta[cid] or "",
                    "evidence": norm_space(x.get("evidence")),
                    "confidence": float(x.get("confidence") or 0.0),
                })

        done_starts.add(start)
        pd.DataFrame(triples).to_csv(ckpt_triples_path, index=False)
        with open(ckpt_state_path, "w", encoding="utf-8") as f:
            json.dump({"done_starts": sorted(done_starts)}, f)

    if len(triples) == 0:
        raise RuntimeError(f"❌ [FAIL-CLOSE] Quá trình trích xuất thu được 0 triples (Tổng lỗi: {len(errors)}). Hãy kiểm tra lại API Key/Model.")
    print(f"✅ Trích xuất thành công {len(triples):,} triples.")
    return pd.DataFrame(triples), pd.DataFrame(errors)

raw_triples_df, extraction_errors_df = run_extraction(extraction_source)
display(raw_triples_df.head())


NER+RE:   0%|          | 0/67 [00:00<?, ?it/s]

✅ Trích xuất thành công 69 triples.


,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence
0,Sineng Electric,Company,USES,EliteSiC,Technology,1a05beb7aa3071be6fd7::c0000,2023-05-16,Sineng Electric will integrate onsemi EliteSiC,0.96
1,GreenPages,Company,ACQUIRED,Zanaris,Company,4bd7afdba71243b0dbcd::c0000,2023-05-02,GreenPages acquired Toronto-based Zanaris,0.97
2,Aeris Communications,Company,PARTNERED_WITH,Ericsson,Company,4f1346392056a403277d::c0000,2022-12-07,Aeris Communications and Ericsson are joining together to create a leader in the fast-growing IoT industry,1.00
3,Ericsson,Company,DEVELOPED,IoT Accelerator,Technology,4f1346392056a403277d::c0000,2022-12-07,Ericsson's IoT Accelerator ... businesses,1.00
4,Ericsson,Company,DEVELOPED,Connected Vehicle Cloud,Technology,4f1346392056a403277d::c0000,2022-12-07,Ericsson's ... Connected Vehicle Cloud businesses,1.00


## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [9]:
#@title 2.2 — Entity resolution
CORP_SUFFIXES = {"inc","incorporated","corp","corporation","ltd","limited","llc","plc","co","company"}
MANUAL_ALIASES = {
    "msft": "Microsoft",
    "microsoft corp": "Microsoft",
    "microsoft corporation": "Microsoft",
    "goog": "Google",
    "googl": "Google",
    "google llc": "Google",
    "meta platforms": "Meta",
    "meta platforms inc": "Meta",
    "aapl": "Apple",
    "apple inc": "Apple",
}

def norm_entity(name):
    s = unicodedata.normalize("NFKC", norm_space(name)).lower()
    s = re.sub(r"[^\w\s\-\.]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def strip_suffix(name):
    toks = norm_entity(name).replace(".", "").split()
    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()
    return " ".join(toks)

def merge_guard(a, b):
    na, nb = strip_suffix(a), strip_suffix(b)
    if na == nb:
        return True
    return SequenceMatcher(None, na, nb).ratio() >= 0.72

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = None

def get_embedder():
    global embedder
    if embedder is None:
        embedder = SentenceTransformer(EMBED_MODEL)
    return embedder

class UF:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a

def build_resolution_map(raw_triples_df, threshold=0.90, top_k=5):
    if raw_triples_df.empty or "source_raw" not in raw_triples_df.columns:
        raise ValueError("❌ [FAIL-CLOSE] `raw_triples_df` rỗng hoặc không đúng schema.")

    mentions = []
    for r in raw_triples_df.itertuples(index=False):
        mentions += [(r.source_type, r.source_raw), (r.target_type, r.target_raw)]

    counts = Counter((t, norm_entity(n)) for t, n in mentions)
    display_name = {}
    for t, n in mentions:
        display_name.setdefault((t, norm_entity(n)), n)

    mapping, audit = {}, []

    for key in counts:
        t, norm = key
        if norm in MANUAL_ALIASES:
            mapping[key] = MANUAL_ALIASES[norm]
            audit.append({
                "type": t, "left": display_name[key],
                "right": MANUAL_ALIASES[norm],
                "similarity": 1.0, "decision": "MERGE_MANUAL"
            })

    for typ in sorted(ALLOWED_NODE_TYPES):
        keys = [k for k in counts if k[0] == typ and k not in mapping]
        if not keys:
            continue
        names = [display_name[k] for k in keys]
        vecs = get_embedder().encode(
            names, batch_size=128, show_progress_bar=False,
            normalize_embeddings=True
        ).astype("float32")

        index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs)
        sims, nbrs = index.search(vecs, min(top_k, len(names)))
        uf = UF(len(names))

        for i in range(len(names)):
            for score, j in zip(sims[i], nbrs[i]):
                if j < 0 or i >= j or float(score) < threshold:
                    continue
                ok = merge_guard(names[i], names[j])
                audit.append({
                    "type": typ, "left": names[i], "right": names[j],
                    "similarity": float(score),
                    "decision": "MERGE_VECTOR" if ok else "REJECT_GUARD"
                })
                if ok:
                    uf.union(i, j)

        groups = defaultdict(list)
        for i in range(len(names)):
            groups[uf.find(i)].append(i)

        for idxs in groups.values():
            best = sorted(
                idxs,
                key=lambda i: (-counts[keys[i]], len(names[i]), names[i].lower())
            )[0]
            canonical = names[best]
            for i in idxs:
                mapping[keys[i]] = canonical

    for key in counts:
        mapping.setdefault(key, display_name[key])

    return mapping, pd.DataFrame(audit)

def canonicalize_triples(raw_df, mapping):
    if raw_df.empty or "source_raw" not in raw_df.columns:
        raise ValueError("❌ [FAIL-CLOSE] `raw_df` rỗng hoặc thiếu cột.")

    df = raw_df.copy()
    def canon(name, typ):
        n = norm_entity(name)
        return mapping.get((typ, n), MANUAL_ALIASES.get(n, name))

    df["source_name"] = [canon(n,t) for n,t in zip(df.source_raw, df.source_type)]
    df["target_name"] = [canon(n,t) for n,t in zip(df.target_raw, df.target_type)]
    df["source_name_norm"] = df.source_name.map(norm_entity)
    df["target_name_norm"] = df.target_name.map(norm_entity)
    df["source_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.source_type, df.source_name_norm)]
    df["target_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.target_type, df.target_name_norm)]
    return df[df.source_id != df.target_id].reset_index(drop=True)

# Tự động phát hiện: nếu có raw_triples_df thì chạy giải quyết thực thể, nếu không thì tải lại từ Neo4j
if "raw_triples_df" in globals() and isinstance(raw_triples_df, pd.DataFrame) and not raw_triples_df.empty:
    entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df)
    triples_df = canonicalize_triples(raw_triples_df, entity_map)
    display(entity_resolution_audit_df.head(20))
else:
    print("ℹ️ `raw_triples_df` không có trong RAM. Đang tải `triples_df` trực tiếp từ Neo4j Cloud...")
    db_triples = run_cypher("""
    MATCH (s:Entity)-[r]->(t:Entity)
    RETURN s.id AS source_id, s.name AS source_name, s.name_norm AS source_name_norm, s.entity_type AS source_type,
           type(r) AS relation,
           t.id AS target_id, t.name AS target_name, t.name_norm AS target_name_norm, t.entity_type AS target_type,
           r.evidence AS evidence, r.published_date AS published_date, r.source_chunk_id AS source_chunk_id, r.confidence AS confidence
    """)
    triples_df = pd.DataFrame(db_triples)
    print(f"✅ Đã nạp thành công {len(triples_df)} triples từ Neo4j AuraDB về DataFrame.")
    display(triples_df.head(10))


ℹ️ `raw_triples_df` không có trong RAM. Đang tải `triples_df` trực tiếp từ Neo4j Cloud...
✅ Đã nạp thành công 69 triples từ Neo4j AuraDB về DataFrame.


,source_id,source_name,source_name_norm,source_type,relation,target_id,target_name,target_name_norm,target_type,evidence,published_date,source_chunk_id,confidence
0,01c0a1b8545f22f61ffe10fc,Squeeze,squeeze,Company,USES,e876dff03ab95567f4d80264,Youtility,youtility,Technology,Squeeze's data analytics and B2C experience with Youtility unique technology allowing switching to be integrated by ...,2023-10-26,e6fe754774c1e887b74f::c0000,0.90
1,02f50203f5a967ac3ebb42be,Ericsson,ericsson,Company,DEVELOPED,3084ee44d67558f56f4da544,Connected Vehicle Cloud,connected vehicle cloud,Technology,Ericsson's ... Connected Vehicle Cloud businesses,2022-12-07,4f1346392056a403277d::c0000,1.00
2,02f50203f5a967ac3ebb42be,Ericsson,ericsson,Company,DEVELOPED,90b846465e0f4548b38b2912,IoT Accelerator,iot accelerator,Technology,Ericsson's IoT Accelerator ... businesses,2022-12-07,4f1346392056a403277d::c0000,1.00
3,0430a89df77e6234ef375307,GreenPages,greenpages,Company,ACQUIRED,d00f157aa6483414b0dba69f,Zanaris,zanaris,Company,GreenPages acquired Toronto-based Zanaris,2023-05-02,4bd7afdba71243b0dbcd::c0000,0.97
4,071dac3f19768ce6d9fdfe2f,ADP,adp,Company,USES,570b60402f62a8953a14c617,Artificial Intelligence / Machine Learning technology,artificial intelligence machine learning technology,Technology,Amazon Web Services helps businesses like ADP ...,2022-12-29,4c8e9044ab0a5a6808a6::c0000,0.94
5,08193f3ca0efe2f71d179f4a,Renovus,renovus,Company,INVESTED_IN,431cca765017665a7961a6ac,Aretum,aretum,Company,Aretum becomes the fourth company backed by Renovus,2023-04-19,c93002837c287180b0c5::c0000,1.00
6,09da89821ca7953b9bd78a87,Kansas City tech company,kansas city tech company,Company,PARTNERED_WITH,bf6198d886439e922a230a09,Notion,notion,Company,A #KansasCity tech company has combined with #Denver-based Notion to become a leader in IoT and smart home services.,2022-12-23,5e56702137be2b9f4d59::c0000,0.95
7,0d38fd81ca44fb425b0da5cc,LeanTaaS,leantaas,Company,ACQUIRED,db47280f083160ab7a469f8a,Hospital IQ,hospital iq,Company,LeanTaaS Acquires Hospital IQ,2023-01-10,99f8ce77e2f393b9bdc5::c0000,1.00
8,0ecbd5177fafb6f25a33707e,Aeris Communications,aeris communications,Company,PARTNERED_WITH,02f50203f5a967ac3ebb42be,Ericsson,ericsson,Company,Aeris Communications and Ericsson are joining together to create a leader in the fast-growing IoT industry,2022-12-07,4f1346392056a403277d::c0000,1.00
9,0f1198d22cf1f786e4405a0d,Associated Press,associated press,Company,PARTNERED_WITH,773eeb9b7cc008bff365fcdd,OpenAI,openai,Company,The Associated Press and OpenAI have reached an agreement to share access to select news content and technology...,2023-07-13,67f61078101e32548302::c0000,0.95


In [10]:
#@title 2.3 — Node table + UNWIND bulk insert
def build_nodes(triples_df):
    rows = []
    for r in triples_df.itertuples(index=False):
        rows += [
            {"id":r.source_id,"name":r.source_name,"name_norm":r.source_name_norm,"type":r.source_type,"alias":getattr(r, "source_raw", r.source_name)},
            {"id":r.target_id,"name":r.target_name,"name_norm":r.target_name_norm,"type":r.target_type,"alias":getattr(r, "target_raw", r.target_name)},
        ]
    df = pd.DataFrame(rows).drop_duplicates()
    grouped = []
    for (i,n,nm,t), sub in df.groupby(["id","name","name_norm","type"]):
        aliases = sorted(set(sub.alias.dropna()))
        grouped.append({
            "id":i,"name":n,"name_norm":nm,"type":t,
            "aliases":aliases,
            "aliases_norm":[norm_entity(a) for a in aliases]
        })
    return pd.DataFrame(grouped)

def insert_nodes_unwind(nodes_df, batch_size=200):
    for start in range(0, len(nodes_df), batch_size):
        batch = nodes_df.iloc[start:start+batch_size].to_dict("records")
        run_cypher("""
        UNWIND $batch AS row
        MERGE (n:Entity {id: row.id})
        SET n.name = row.name,
            n.name_norm = row.name_norm,
            n.entity_type = row.type,
            n.aliases = row.aliases,
            n.aliases_norm = row.aliases_norm
        WITH n, row
        CALL apoc.create.addLabels(n, [row.type]) YIELD node
        RETURN count(node)
        """, batch=batch)

def insert_edges_unwind(triples_df, batch_size=200):
    for rel in ALLOWED_RELATIONS:
        sub = triples_df[triples_df.relation == rel]
        if sub.empty:
            continue
        for start in range(0, len(sub), batch_size):
            batch = sub.iloc[start:start+batch_size].to_dict("records")
            q = f"""
            UNWIND $batch AS row
            MATCH (s:Entity {{id: row.source_id}})
            MATCH (t:Entity {{id: row.target_id}})
            MERGE (s)-[r:{rel}]->(t)
            SET r.source_chunk_id = row.source_chunk_id,
                r.published_date = row.published_date,
                r.evidence = row.evidence,
                r.confidence = row.confidence
            """
            run_cypher(q, batch=batch)

# Kiểm tra dữ liệu hiện có trong Neo4j
existing_node_count = run_cypher("MATCH (n:Entity) RETURN count(n) AS n")[0]["n"]
if existing_node_count > 0:
    print(f"✅ Neo4j đã có sẵn {existing_node_count} nodes và {len(triples_df)} edges. Bỏ qua bulk insert để tiết kiệm thời gian.")
    nodes_df = pd.DataFrame(run_cypher("""
    MATCH (n:Entity)
    RETURN n.id AS id, n.name AS name, n.name_norm AS name_norm, n.entity_type AS type, n.aliases AS aliases, n.aliases_norm AS aliases_norm
    """))
else:
    nodes_df = build_nodes(triples_df)
    insert_nodes_unwind(nodes_df)
    insert_edges_unwind(triples_df)
    print(f"✅ Đã nạp thành công {len(nodes_df)} nodes và {len(triples_df)} edges vào Neo4j.")

display(nodes_df.head(10))


✅ Neo4j đã có sẵn 121 nodes và 69 edges. Bỏ qua bulk insert để tiết kiệm thời gian.


,id,name,name_norm,type,aliases,aliases_norm
0,00b6c835385fb8530abe1bec,Stellantis,stellantis,Company,[Stellantis],[stellantis]
1,01c0a1b8545f22f61ffe10fc,Squeeze,squeeze,Company,[Squeeze],[squeeze]
2,02f50203f5a967ac3ebb42be,Ericsson,ericsson,Company,[Ericsson],[ericsson]
3,0430a89df77e6234ef375307,GreenPages,greenpages,Company,[GreenPages],[greenpages]
4,071dac3f19768ce6d9fdfe2f,ADP,adp,Company,[ADP],[adp]
5,08193f3ca0efe2f71d179f4a,Renovus,renovus,Company,[Renovus],[renovus]
6,09da89821ca7953b9bd78a87,Kansas City tech company,kansas city tech company,Company,[Kansas City tech company],[kansas city tech company]
7,0c104aea0811fd35b52eaf70,General Services Administration,general services administration,Company,[General Services Administration],[general services administration]
8,0d38fd81ca44fb425b0da5cc,LeanTaaS,leantaas,Company,[LeanTaaS],[leantaas]
9,0ecbd5177fafb6f25a33707e,Aeris Communications,aeris communications,Company,[Aeris Communications],[aeris communications]


In [11]:
#@title 2.4 — Sanity checks
def graph_checks():
    invalid = run_cypher("""
    MATCH ()-[r]->()
    WHERE r.source_chunk_id IS NULL OR r.published_date IS NULL
    RETURN count(r) AS n
    """)[0]["n"]

    counts = {
        "nodes": run_cypher("MATCH (n:Entity) RETURN count(n) AS n")[0]["n"],
        "edges": run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"],
        "invalid_provenance_edges": invalid,
    }
    print(counts)
    assert invalid == 0

    top = pd.DataFrame(run_cypher("""
    MATCH (n:Entity)
    OPTIONAL MATCH (n)-[r]-()
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT 15
    """))
    display(top)
    return counts, top

graph_counts, top_degree_df = graph_checks()

{'nodes': 121, 'edges': 69, 'invalid_provenance_edges': 0}


,id,name,type,degree
0,fb0f4df56fab164ec48722f0,Microsoft,Company,4
1,23d7ceb58c062d83135817ba,Intelligent Technical Solutions,Company,3
2,9dc9c71ce167faf79f0752f4,Dubai Electricity and Water Authority,Company,3
3,02f50203f5a967ac3ebb42be,Ericsson,Company,3
4,9fed1582022a037a3cfb3d66,Christopher Kirchner,Person,2
5,570b60402f62a8953a14c617,Artificial Intelligence / Machine Learning technology,Technology,2
6,d0eaaccf04669315e412ad07,DI,Company,2
7,4af878a52d972da7f5af3015,Slync,Company,2
8,43c9626ecf3e924d3289896d,Samsung Electronics Co. Ltd.,Company,2
9,ce493f3efd776c3a90d1c029,Crexendo,Company,2


# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [12]:
#@title 3.1 — Flat RAG
flat_index = None
flat_store = None
entity_match_vectors = None
entity_match_store = None

def build_flat_index(chunks_df):
    global flat_index, flat_store
    vecs = get_embedder().encode(
        chunks_df.text.fillna("").tolist(),
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True
    ).astype("float32")

    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print("Flat vectors:", flat_index.ntotal)

def retrieve_flat_context(query, k=6):
    qv = get_embedder().encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score":float(score), "chunk_id":r.chunk_id,
            "published_date":r.published_date, "text":r.text
        })

    df = pd.DataFrame(rows)
    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df

build_flat_index(chunks_df)

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Flat vectors: 5004


## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [22]:
#@title 3.2 — Seed matching
SEED_SYSTEM = """
You are an entity extractor for a knowledge graph.
Extract useful seed entities from the question.
Allowed types: Company, Person, Technology.
Return strict JSON format with key 'seeds'.
Example: {"seeds": [{"name": "Apple", "type": "Company"}]}
""".strip()

def extract_seeds(query):
    try:
        prompt = f"""
Question: {query}
Extract seed entities. Return strict JSON format:
{{"seeds": [{{"name": "...", "type": "Company|Person|Technology|null"}}]}}
""".strip()
        obj, _ = groq_json(SEED_SYSTEM, prompt)
        return [
            {"name": norm_space(x.get("name")),
             "type": x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
            for x in obj.get("seeds", [])
            if norm_space(x.get("name"))
        ]
    except Exception as e:
        print(f"⚠️ [extract_seeds fallback] Không bóc tách được seeds từ query: '{query[:50]}...' ({e})")
        return []

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    names = nodes_df.name.tolist()
    entity_match_store = nodes_df[["id","name","type"]].to_dict("records")
    entity_match_vectors = get_embedder().encode(
        names, batch_size=128, show_progress_bar=False,
        normalize_embeddings=True
    ).astype("float32")
    return len(names)

def fuzzy_match_entity(name, typ=None, top_k=3, min_sim=0.66):
    if entity_match_vectors is None:
        return []
    qv = get_embedder().encode([name], normalize_embeddings=True).astype("float32")
    sims = (entity_match_vectors @ qv.T).squeeze(1)
    idxs = np.argsort(-sims)[:top_k]
    hits = []
    for idx in idxs:
        score = float(sims[idx])
        if score < min_sim:
            continue
        item = entity_match_store[idx]
        if typ and item["type"] != typ:
            continue
        hits.append({**item, "similarity": score})
    return hits

def match_seeds(query, fuzzy_threshold=0.66):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Entity)
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"])

        if exact:
            matched += exact
        else:
            matched += fuzzy_match_entity(seed["name"], seed["type"], min_sim=fuzzy_threshold)

    # De-duplicate
    seen, out = set(), []
    for m in matched:
        if m["id"] not in seen:
            seen.add(m["id"])
            out.append(m)
    return out

build_entity_matcher(nodes_df)


121

In [16]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE = 100
SUPER_NODE_EDGE_CAP = 50
GLOBAL_EDGE_CAP = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Entity {id:$id})
    OPTIONAL MATCH (n)-[r]-()
    RETURN count(r) AS degree
    """, id=node_id)[0]["degree"])

def recent_edges(node_id, limit):
    return run_cypher("""
    MATCH (n:Entity {id:$id})
    MATCH (n)-[r]-(m:Entity)
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      m.id AS neighbor_id
    ORDER BY coalesce(r.published_date,'') DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit))

def textualize(edges):
    edges = sorted(edges, key=lambda e:e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False):
    seeds = match_seeds(query)
    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"],e["relation"],e["target_id"],e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
        }
    }
    return out if return_debug else out["context"]

In [17]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = """
Answer only from supplied context.
Be concise but complete. Do not invent facts.
Cite provenance inline as [chunk_id=...] whenever possible.
If evidence is insufficient or conflicting, say so.
""".strip()

def generate_answer(question, context):
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0 = time.perf_counter()
    text, usage = groq_chat(
        [{"role":"system","content":ANSWER_SYSTEM},
         {"role":"user","content":prompt}],
        model=GROQ_MODEL
    )
    return {
        "answer": text.strip(),
        "latency_s": time.perf_counter()-t0,
        "total_tokens": usage.get("total_tokens"),
    }

def answer_flat_rag(question):
    context, retrieved = retrieve_flat_context(question, k=6)
    out = generate_answer(question, context)
    out.update({"context":context,"retrieved":retrieved})
    return out

def answer_graph_rag(question):
    g = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs})
    return out

# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE (75–105')

## Golden schema
`id`, `group`, `question`, `reference_answer`, optional `reference_evidence`.

Notebook có 5 câu starter. Các câu phụ thuộc data dump phải điền gold answer thật trước final evaluation.

In [18]:
#@title 4.1 — 5 câu Golden starter
GOLDEN_PATH = "data/golden_dataset.csv"

starter_golden = pd.DataFrame([
    {
        "id":"G01","group":"factoid",
        "question":"Who was the CEO of Hugging Face in 2023?",
        "reference_answer":"Clément Delangue",
        "reference_evidence":"Validate against instructor dump."
    },
    {
        "id":"G02","group":"multi-hop",
        "question":"Which startups were founded by former Microsoft employees and later received investment from Google?",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G03","group":"cross-doc",
        "question":"Compare the direction of AI-related investments by Meta and Apple during 2023 using evidence from multiple articles.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G04","group":"multi-hop",
        "question":"Find a company invested in by a major technology company that also developed a named AI technology; identify both relations and dates.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G05","group":"cross-doc",
        "question":"Identify one technology connected to the same company in at least two news chunks and summarize how the relationship changed over time.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
])

golden_df = pd.read_csv(GOLDEN_PATH) if Path(GOLDEN_PATH).exists() else starter_golden.copy()
display(golden_df)

def validate_golden(df, require_answers=True):
    required = {"id","group","question","reference_answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns: {required-set(df.columns)}")
    if require_answers and df.reference_answer.fillna("").str.strip().eq("").any():
        display(df[df.reference_answer.fillna("").str.strip().eq("")][["id","question"]])
        raise ValueError("Điền reference_answer trước final evaluation.")
    print("✅ Golden Dataset valid.")

,id,group,question,reference_answer,reference_evidence
0,G5000-01,multi-hop,Reconstruct the Aeris–Ericsson IoT transaction across the available reports: which Ericsson businesses moved to Aeri...,"Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses, together with related assets, were to be transfer...",row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...
1,G5000-02,cross-doc,"Did the first two Aeris/Ericsson reports describe a completed acquisition or a planned transfer, and what later evid...",The first reports describe a planned transaction: Aeris was to acquire Ericsson's IoT Accelerator and Connected Vehi...,row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...
2,G5000-03,factoid,"After the Aeris–Ericsson IoT deal progressed, how many IoT devices, enterprises, and countries were cited in the lat...","More than 100 million IoT devices, 9,000 enterprises, and 190 countries.",row 935 (2023-01-18 22:37:00): A Leap in Connectivity: Aeris Acquires Technologies from Ericsson to Support Cellular...
3,G5000-04,cross-doc,"Which two named Ericsson IoT businesses recur across multiple reports of the Aeris transaction, and why should Graph...",The recurring businesses are Ericsson IoT Accelerator and Connected Vehicle Cloud. The reports describe the same Aer...,row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...
4,G5000-05,multi-hop,"Starting from Ericsson, follow the graph to the acquirer and then to the reported IoT reach. What path and scale sho...",Ericsson -> (IoT Accelerator and Connected Vehicle Cloud transferred/acquired by) Aeris -> supports/connects more th...,row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 935 (2023-01-18 22:37:00): A Leap in...
5,G5000-06,multi-hop,Trace ServiceNow's generative-AI product/partner evolution from May through July 2023: what partnership began in May...,"In May, ServiceNow partnered with NVIDIA to build enterprise-grade generative AI for workflow automation. In June, S...",row 746 (2023-05-17 18:58:00): ServiceNow and NVIDIA Announce Partnership to Build Generative AI Across Enterprise I...
6,G5000-07,cross-doc,How did ServiceNow's July generative-AI expansion differ between its platform features and its ecosystem program wit...,ServiceNow expanded the Now Assist feature set with case summarization and text-to-code inside its workflow platform...,row 1435 (2023-07-26 20:30:00): ServiceNow Expands Generative AI Capabilities With Case Summarization and Text-to-Co...
7,G5000-08,multi-hop,"Which external organizations are connected to ServiceNow's generative-AI efforts in the selected data, and what dist...",NVIDIA is ServiceNow's generative-AI technology partner and later co-launch partner for AI Lighthouse; Accenture joi...,row 746 (2023-05-17 18:58:00): ServiceNow and NVIDIA Announce Partnership to Build Generative AI Across Enterprise I...
8,G5000-09,cross-doc,Two June 13 reports describe Now Assist for Virtual Agent. What should a deduplicated graph store as the core event ...,A single ServiceNow product event: Now Assist for Virtual Agent extends ServiceNow's generative-AI capabilities for ...,row 1742 (2023-06-13 13:50:00): ServiceNow Unveils Generative AI Solution With Real-Time Conversational Experiences ...
9,G5000-10,multi-hop,What sequence shows ServiceNow moving from domain-specific generative-AI exploration to concrete workflow features?,"At Knowledge 2023, ServiceNow emphasized domain-specific generative-AI/LLM use cases; in June it introduced Now Assi...",row 752 (2023-05-18 00:32:00): Knowledge 2023 - ServiceNow looks beyond general purpose generative AI to domain-spec...


In [19]:
#@title 4.2 — LLM-as-a-Judge
JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.
Score 1-5:
- comprehensiveness
- faithfulness to supplied candidate context
- multi_hop_reasoning accuracy
Use the reference answer as correctness anchor.
Return strict JSON only.
""".strip()

def judge_json(system, user):
    current_judge_model = os.environ.get("JUDGE_MODEL", JUDGE_MODEL)
    current_provider = os.environ.get("JUDGE_PROVIDER", JUDGE_PROVIDER)

    if not current_judge_model:
        raise RuntimeError("❌ [FAIL-CLOSE] Thiếu JUDGE_MODEL.")

    if current_provider == "groq":
        return groq_json(system, user, model=current_judge_model)[0]

    if current_provider == "openai":
        if not OPENAI_API_KEY:
            raise RuntimeError("❌ [FAIL-CLOSE] Thiếu OPENAI_API_KEY.")
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        resp = client.chat.completions.create(
            model=current_judge_model,
            messages=[{"role":"system","content":system},
                      {"role":"user","content":user}],
            temperature=0.0,
            response_format={"type":"json_object"}
        )
        return parse_json_object(resp.choices[0].message.content)

    raise ValueError("JUDGE_PROVIDER must be openai or groq.")

def judge_answer(question, reference, answer, context):
    prompt = f"""
QUESTION:
{question}

REFERENCE:
{reference}

CANDIDATE:
{answer}

CANDIDATE CONTEXT:
{context[:18000]}

Return:
{{
 "comprehensiveness":1,
 "faithfulness":1,
 "multi_hop_reasoning":1,
 "rationale":"2-5 sentences"
}}
"""
    obj = judge_json(JUDGE_SYSTEM, prompt)
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        out[k] = max(1, min(5, int(obj.get(k,1))))
    out["rationale"] = norm_space(obj.get("rationale"))
    return out


In [43]:
#@title 4.3 — Evaluation runner + checkpoint (Tự động chạy tiếp)
CHECKPOINT = "outputs/graphrag_eval_checkpoint.csv"

def run_evaluation(golden_df, resume=True):
    done_map = {}
    if resume and os.path.exists(CHECKPOINT):
        try:
            ckpt_df = pd.read_csv(CHECKPOINT)
            if not ckpt_df.empty and "id" in ckpt_df.columns:
                done_map = {r["id"]: r.to_dict() for _, r in ckpt_df.iterrows()}
                print(f"🔄 Đã tìm thấy checkpoint với {len(done_map)}/{len(golden_df)} câu đã xong. Đang chạy tiếp các câu còn lại...")
        except Exception as e:
            print(f"Không đọc được checkpoint ({e}), chạy mới...")

    rows = list(done_map.values())
    for q in tqdm(golden_df.itertuples(index=False), total=len(golden_df), desc="Evaluation"):
        if q.id in done_map:
            continue

        flat = answer_flat_rag(q.question)
        graph = answer_graph_rag(q.question)

        jf = judge_answer(q.question, q.reference_answer, flat["answer"], flat["context"])
        jg = judge_answer(q.question, q.reference_answer, graph["answer"], graph["context"])

        row_data = {
            "id":q.id, "group":q.group, "question":q.question,
            "reference_answer":q.reference_answer,
            "flat_answer":flat["answer"], "graph_answer":graph["answer"],
            "flat_comprehensiveness":jf["comprehensiveness"],
            "graph_comprehensiveness":jg["comprehensiveness"],
            "flat_faithfulness":jf["faithfulness"],
            "graph_faithfulness":jg["faithfulness"],
            "flat_multi_hop_reasoning":jf["multi_hop_reasoning"],
            "graph_multi_hop_reasoning":jg["multi_hop_reasoning"],
            "flat_latency_s":flat["latency_s"],
            "graph_latency_s":graph["latency_s"],
            "flat_total_tokens":flat.get("total_tokens"),
            "graph_total_tokens":graph.get("total_tokens"),
            "flat_judge_rationale":jf["rationale"],
            "graph_judge_rationale":jg["rationale"],
            "graph_supernode_events":len(
                graph["graph_debug"]["diagnostics"].get("supernode_events",[])
            )
        }
        rows.append(row_data)
        os.makedirs(os.path.dirname(CHECKPOINT) or ".", exist_ok=True)
        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)
    return pd.DataFrame(rows)

validate_golden(golden_df, require_answers=True)
eval_results_df = run_evaluation(golden_df)
display(eval_results_df.head(10))


✅ Golden Dataset valid.
🔄 Đã tìm thấy checkpoint với 49/50 câu đã xong. Đang chạy tiếp các câu còn lại...


Evaluation:   0%|          | 0/50 [00:00<?, ?it/s]

⚠️ [Model switch] 'openai/gpt-oss-20b' het han muc/khong kha dung: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01kz809fpcehdt02x9355xwc7x` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199625, Requested 636. Please try again in 1m52.752s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}. Chuyen sang model ke tiep trong chain...
⚠️ [Model switch] 'openai/gpt-oss-120b' het han muc/khong kha dung: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kz809fpcehdt02x9355xwc7x` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 200000, Requested 446. Please try again in 3m12.672s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}. Chuyen sang m

,id,group,question,reference_answer,flat_answer,graph_answer,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,graph_faithfulness,flat_multi_hop_reasoning,graph_multi_hop_reasoning,flat_latency_s,graph_latency_s,flat_total_tokens,graph_total_tokens,flat_judge_rationale,graph_judge_rationale,graph_supernode_events
0,G5000-01,multi-hop,Reconstruct the Aeris–Ericsson IoT transaction across the available reports: which Ericsson businesses moved to Aeri...,"Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses, together with related assets, were to be transfer...",Aeris acquired Ericsson’s **IoT Accelerator** and **Connected Vehicle Cloud** businesses (and related assets) [chunk...,The transaction transferred **Ericsson’s IoT Accelerator and Connected Vehicle Cloud businesses** (and their related...,5,5,5,5,5,5,1.024548,0.668989,974,933,The candidate answer accurately lists the Ericsson businesses transferred to Aeris (IoT Accelerator and Connected Ve...,The candidate answer accurately lists the Ericsson businesses (IoT Accelerator and Connected Vehicle Cloud) transfer...,0
1,G5000-02,cross-doc,"Did the first two Aeris/Ericsson reports describe a completed acquisition or a planned transfer, and what later evid...",The first reports describe a planned transaction: Aeris was to acquire Ericsson's IoT Accelerator and Connected Vehi...,The first two reports (dated 2022‑12‑07 and 2023‑01‑10) described a **planned transfer** of Ericsson’s IoT Accelerat...,The first two reports (the 2022‑12‑07 and 2023‑01‑10 pieces) framed the deal as a **planned transfer** of Ericsson’s...,5,5,5,5,5,5,0.730671,1.086661,1021,1110,The candidate answer correctly identifies that the first two reports described a planned transfer and that a later J...,The candidate answer accurately captures the key points of the reference: it identifies that the first two reports d...,0
2,G5000-03,factoid,"After the Aeris–Ericsson IoT deal progressed, how many IoT devices, enterprises, and countries were cited in the lat...","More than 100 million IoT devices, 9,000 enterprises, and 190 countries.","The later connectivity report cited that Aeris would connect **more than 100 million IoT devices for 9,000 enterpris...","The later connectivity report cited that Aeris would connect **more than 100 million IoT devices for 9,000 enterpris...",5,5,5,5,5,5,7.654109,7.038258,799,825,"The candidate answer correctly lists all three figures—more than 100 million IoT devices, 9,000 enterprises, and 190...","The candidate answer correctly lists all three figures—more than 100 million IoT devices, 9,000 enterprises, and 190...",0
3,G5000-04,cross-doc,"Which two named Ericsson IoT businesses recur across multiple reports of the Aeris transaction, and why should Graph...",The recurring businesses are Ericsson IoT Accelerator and Connected Vehicle Cloud. The reports describe the same Aer...,The reports consistently refer to **Ericsson’s “IoT Accelerator”** and **“Connected Vehicle Cloud”** businesses as t...,The two Ericsson IoT businesses that appear in several Aeris‑transaction reports are **IoT Accelerator** and **Conne...,5,5,5,5,5,5,1.982049,0.964743,953,1061,The candidate correctly identifies both Ericsson IoT Accelerator and Connected Vehicle Cloud as the recurring busine...,The candidate correctly identifies Ericsson IoT Accelerator and Connected Vehicle Cloud as the recurring businesses ...,0
4,G5000-05,multi-hop,"Starting from Ericsson, follow the graph to the acquirer and then to the reported IoT reach. What path and scale sho...",Ericsson -> (IoT Accelerator and Connected Vehicle Cloud transferred/acquired by) Aeris -> supports/connects more th...,**Path**: Ericsson → Aeris Communications → 100 million IoT devices (acquired IoT Accelerator & Connected Vehicle Cl...,**Path:** Ericsson → Aeris \n**Scale:** > 100 million IoT devices (across 9 000 enterprises in 190 countries)【chunk...,5,5,5,5,5,5,7.200391,0.670528,898,653,The candidate correctly identifies

In [44]:
#@title 4.4 — Comparison table + export
def comparison_table(eval_df):
    metric_map = {
        "Comprehensiveness":("flat_comprehensiveness","graph_comprehensiveness"),
        "Faithfulness":("flat_faithfulness","graph_faithfulness"),
        "Multi-hop reasoning":("flat_multi_hop_reasoning","graph_multi_hop_reasoning"),
        "Latency (s)":("flat_latency_s","graph_latency_s"),
        "Token usage":("flat_total_tokens","graph_total_tokens"),
    }

    rows = []
    for group, g in eval_df.groupby("group"):
        for metric, (fc,gc) in metric_map.items():
            f = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()

            if metric in {"Latency (s)","Token usage"}:
                comment = "Flat RAG thường rẻ/nhanh hơn." if f < gr else "GraphRAG không đắt hơn trong sample này."
            else:
                delta = gr - f
                if delta >= .75:
                    comment = "GraphRAG cải thiện rõ; kiểm tra rationale và provenance."
                elif delta <= -.5:
                    comment = "Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu."
                else:
                    comment = "Hai phương pháp gần nhau."

            rows.append({
                "Loại câu hỏi":group, "Metric":metric,
                "Flat RAG":round(f,3) if pd.notna(f) else np.nan,
                "GraphRAG":round(gr,3) if pd.notna(gr) else np.nan,
                "Nhận xét phân tích":comment
            })
    return pd.DataFrame(rows)

comparison_df = comparison_table(eval_results_df)
display(comparison_df)
eval_results_df.to_csv("outputs/graphrag_eval_results.csv", index=False)
comparison_df.to_csv("outputs/graphrag_vs_flatrag_summary.csv", index=False)

,Loại câu hỏi,Metric,Flat RAG,GraphRAG,Nhận xét phân tích
0,cross-doc,Comprehensiveness,3.455,3.227,Hai phương pháp gần nhau.
1,cross-doc,Faithfulness,3.500,3.409,Hai phương pháp gần nhau.
2,cross-doc,Multi-hop reasoning,3.545,3.182,Hai phương pháp gần nhau.
3,cross-doc,Latency (s),6.034,4.869,GraphRAG không đắt hơn trong sample này.
4,cross-doc,Token usage,1003.318,900.455,GraphRAG không đắt hơn trong sample này.
5,factoid,Comprehensiveness,5.000,4.400,Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu.
6,factoid,Faithfulness,5.000,4.400,Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu.
7,factoid,Multi-hop reasoning,5.000,4.400,Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu.
8,factoid,Latency (s),9.700,13.761,Flat RAG thường rẻ/nhanh hơn.
9,factoid,Token usage,769.800,699.000,GraphRAG không đắt hơn trong sample này.


# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [45]:
#@title 5.1 — Super-node check + entity audit
def test_supernode_policy():
    rows = run_cypher("""
    MATCH (n:Entity)-[r]-()
    WITH n, count(r) AS degree
    ORDER BY degree DESC LIMIT 1
    RETURN n.id AS id, n.name AS name, degree
    """)
    if not rows:
        print("Graph empty.")
        return

    n = rows[0]
    limit = 50 if n["degree"] > SUPER_NODE_DEGREE else 1000
    edges = recent_edges(n["id"], limit)
    print(n, "fetched=", len(edges))
    if n["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= 50
        print("✅ Super-node cap OK.")

def show_resolution_audit(audit_df):
    if audit_df.empty:
        print("No audit rows.")
        return
    display(
        audit_df.sort_values("similarity", ascending=False).head(30)
    )
    print("High-similarity rejected pairs:")
    display(
        audit_df[audit_df.decision=="REJECT_GUARD"]
        .sort_values("similarity", ascending=False)
        .head(20)
    )

test_supernode_policy()
show_resolution_audit(entity_resolution_audit_df)

{'id': 'fb0f4df56fab164ec48722f0', 'name': 'Microsoft', 'degree': 4} fetched= 4


NameError: name 'entity_resolution_audit_df' is not defined

## 5.2 — Thuyết minh kỹ thuật: học viên tự điền

1. Coreference sai ở tình huống nào?
2. Entity threshold bao nhiêu, vì sao?
3. Candidate nào similarity cao nhưng không nên merge?
4. Top 3 super-node và degree?
5. Vì sao ưu tiên edge mới nhất có thể đúng/sai?
6. Flat RAG thắng nhóm nào?
7. GraphRAG thắng nhóm nào?
8. Latency/token trade-off?
9. AI Coding Agent đề xuất gì mà bạn **không dùng**, vì sao?
10. Scale 350MB: bottleneck đầu tiên là gì?

# 🎁 BONUS

## A — Low-level / High-level
Tạo local entities và high-level topics/community reports; query router chọn tầng retrieval.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. LLM summarize community,
5. query global trên reports.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [ ]:
#@title Bonus — NetworkX community fallback
import networkx as nx

def build_communities(limit_edges=20000):
    edge_df = pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r]->(b:Entity)
    RETURN a.id AS source, b.id AS target
    LIMIT $limit
    """, limit=int(limit_edges)))

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = nx.algorithms.community.greedy_modularity_communities(G)

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id":node_id,"community_id":int(cid)} for node_id in members]

    for b in batches(rows, 1000):
        run_cypher("""
        UNWIND $rows AS row
        MATCH (n:Entity {id:row.id})
        SET n.community_id=row.community_id
        """, rows=b)

    return pd.DataFrame(rows)

community_df = build_communities()

In [ ]:
#@title Bonus — Self-correction scaffold
SUFFICIENCY_SYSTEM = """
Decide whether the supplied retrieval context is sufficient to answer the question faithfully.
Do not answer the question. Return strict JSON only.
""".strip()

def context_sufficient(question, context):
    obj, _ = groq_json(
        SUFFICIENCY_SYSTEM,
        f"""QUESTION: {question}
CONTEXT:
{context[:16000]}
Return {{"sufficient":true,"missing":"..."}}"""
    )
    return bool(obj.get("sufficient")), norm_space(obj.get("missing"))

def self_correcting_context(question):
    g2 = retrieve_graph_context(question, 2, 50, True)
    ok, missing = context_sufficient(question, g2["context"])
    if ok:
        return {"route":"hop2","context":g2["context"],"missing":""}

    g3 = retrieve_graph_context(question, 3, 50, True)
    ok, missing2 = context_sufficient(question, g3["context"])
    if ok:
        return {"route":"hop3","context":g3["context"],"missing":missing}

    flat, _ = retrieve_flat_context(question, k=8)
    return {
        "route":"hop3+vector",
        "context":f"=== GRAPH ===\n{g3['context']}\n\n=== VECTOR ===\n{flat}",
        "missing":missing2
    }

# ✅ RUBRIC

- **30% Chạy được code:** graph nạp thành công, schema đúng, xuất bảng.
- **30% Failure modes:** xử lý ít nhất 2/3 vấn đề Super-node, Entity Resolution, Coreference.
- **20% Evaluation:** chạy hết Golden Dataset, phân tích hợp lý.
- **20% Thuyết minh:** giải thích kiến trúc và cách kiểm soát AI Coding Agent.

## Submission checklist
- [ ] Neo4j connected
- [ ] Dedup/chunking đã chạy
- [ ] Coreference spot-check
- [ ] Entity resolution audit
- [ ] `UNWIND` bulk insert
- [ ] 0 edge thiếu provenance
- [ ] Flat RAG chạy
- [ ] GraphRAG chạy
- [ ] Super-node check
- [ ] Golden Dataset có gold answers thật
- [ ] Evaluation chạy hết
- [ ] Export results + summary CSV
- [ ] Thuyết minh kỹ thuật
- [ ] Bonus (nếu có) có định lượng trước/sau